# Regression Exploration

A persistent record of the data, split, feature, and TPOT checks used to set up the classifier experiments.

In [1]:
from pathlib import Path
import importlib.util
import inspect
import sys

import pandas as pd

working_dir = Path.cwd()
repo_root = working_dir.parent if working_dir.name == 'regression' else working_dir
regression_dir = repo_root / 'regression'
sys.path.insert(0, str(regression_dir))

from data import load_data, load_stopwords, make_groups
from evaluation import split_data
from model import build_vectorizer

data = load_data(repo_root / 'data' / 'processed' / 'headlines.csv')
data['group'] = make_groups(data)
labels = sorted(data['label'].unique())
train_indices, test_indices = split_data(data)
train = data.iloc[train_indices]
test = data.iloc[test_indices]

print(f'Rows: {len(data)} | columns: {list(data.columns)}')
print(f'Groups: {data["group"].nunique()} | train: {len(train)} | test: {len(test)}')
display(data['label'].value_counts().rename('rows').to_frame())
display(pd.DataFrame({
    'train': train['label'].value_counts(),
    'test': test['label'].value_counts(),
}).fillna(0).astype(int).reindex(labels, fill_value=0))

Rows: 455 | columns: ['outlet', 'date', 'headline', 'url', 'summary', 'predicted_label', 'predicted_confidence', 'predicted_probs', 'model_version', 'label', 'comment', 'labelled_by', 'labelled_at', 'cluster_id', 'group']
Groups: 369 | train: 323 | test: 132


,rows
label,
vilifying_opponents,245
nothing,84
vilifying_neighbours,69
personality_cult,38
vilifying_eu,19


,train,test
label,,
nothing,62,22
personality_cult,31,7
vilifying_eu,7,12
vilifying_neighbours,49,20
vilifying_opponents,174,71


## Shared TF-IDF Features

Fit the vocabulary on training headlines only, then transform held-out headlines. This keeps held-out text from influencing the feature vocabulary.

In [2]:
vectorizer = build_vectorizer()
X_train = vectorizer.fit_transform(train['headline'])
X_test = vectorizer.transform(test['headline'])
feature_names = vectorizer.get_feature_names_out()

print(f'Train matrix: {type(X_train).__name__}, shape={X_train.shape}')
print(f'Test matrix:  {type(X_test).__name__}, shape={X_test.shape}')
print(f'Vocabulary size: {len(feature_names)}')

sample_index = 0
print(train.iloc[sample_index]['headline'])
pd.Series(
    X_train[sample_index].toarray().ravel(),
    index=feature_names,
).nlargest(10).to_frame('tfidf')

Train matrix: csr_matrix, shape=(323, 7339)
Test matrix:  csr_matrix, shape=(132, 7339)
Vocabulary size: 7339
NISMO NI SUMNJALI DA ĆE OVAKVA "GROMADA" STATI IZA BLOKADERA! Crnogorac koji podržava Srdanovića mrzi SPC, slavi lažnu državu Kosovo, divljao po Vatikanu (FOTO)


,tfidf
blokadera crnogorac,0.157254
crnogorac,0.157254
crnogorac koji,0.157254
divljao,0.157254
divljao po,0.157254
državu kosovo,0.157254
gromada,0.157254
gromada stati,0.157254
iza blokadera,0.157254
koji podržava,0.157254


## TPOT Availability

Check the active notebook kernel before building the TPOT experiment; package APIs differ across TPOT releases.

In [3]:
tpot_spec = importlib.util.find_spec('tpot')
if tpot_spec is None:
    print('TPOT is not installed in this notebook kernel.')
else:
    try:
        import tpot
        print(f'TPOT version: {getattr(tpot, "__version__", "unknown")}')
        print('TPOTClassifier signature:')
        print(inspect.signature(tpot.TPOTClassifier))
    except Exception as error:
        print(f'TPOT is present but could not be imported: {type(error).__name__}: {error}')

/Users/jtakalai/python/propaganda-radar/regression/.venv/lib/python3.13/site-packages/stopit/__init__.py:10: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/Users/jtakalai/python/propaganda-radar/regression/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


TPOT version: 1.1.0
TPOTClassifier signature:
(search_space='linear', scorers=['roc_auc_ovr'], scorers_weights=[1], cv=10, other_objective_functions=[], other_objective_functions_weights=[], objective_function_names=None, bigger_is_better=True, categorical_features=None, memory=None, preprocessing=False, max_time_mins=60, max_eval_time_mins=10, n_jobs=1, validation_strategy='none', validation_fraction=0.2, early_stop=None, warm_start=False, periodic_checkpoint_folder=None, verbose=2, memory_limit=None, client=None, random_state=None, allow_inner_classifiers=None, **tpotestimator_kwargs)


## Fast TPOT Smoke Test

A very short search confirms that the current kernel, sparse feature matrix, TPOT API, and label decoding work end to end. This is only a compatibility check, not a meaningful model comparison.

In [5]:
from tpot import TPOTClassifier
from evaluation import evaluate

fast_est = TPOTClassifier(
    search_space="linear",
    scorers=["balanced_accuracy"],
    scorers_weights=[1.0],
    cv=2,
    n_jobs=1,
    max_time_mins=0.5,
    max_eval_time_mins=0.5,
    verbose=0,
    random_state=42,
)
fast_est.fit(X_train, train["label"])
fast_predictions = fast_est.label_encoder_.inverse_transform(
    fast_est.predict(X_test).astype(int)
)
fast_metrics = evaluate(test["label"], fast_predictions, labels)
print(f"Fast-search pipeline: {fast_est.fitted_pipeline_}")
display(
    pd.DataFrame(fast_metrics["report"]).T[
        ["precision", "recall", "f1-score", "support"]
    ].round(3)
)

/Users/jtakalai/python/propaganda-radar/regression/.venv/lib/python3.13/site-packages/distributed/node.py:195: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 54074 instead
  warnings.warn(
/Users/jtakalai/python/propaganda-radar/regression/.venv/lib/python3.13/site-packages/tpot/tpot_estimator/estimator.py:537: UserWarning: Labels are not encoded as ints from 0 to N. For compatibility with some classifiers such as sklearn, TPOT has encoded y with the sklearn LabelEncoder. When using pipelines outside the main TPOT estimator class, you can encode the labels with est.label_encoder_
  warnings.warn("Labels are not encoded as ints from 0 to N. For compatibility with some classifiers such as sklearn, TPOT has encoded y with the sklearn LabelEncoder. When using pipelines outside the main TPOT estimator class, you can encode the labels with est.label_encoder_")
/Users/jtakalai/python/propaganda-radar/regression/.venv/lib/p

Fast-search pipeline: Pipeline(steps=[('normalizer', Normalizer(norm=np.str_('max'))),
                ('rfe',
                 RFE(estimator=ExtraTreesClassifier(bootstrap=np.True_,
                                                    criterion=np.str_('gini'),
                                                    max_features=0.6818264375152,
                                                    min_samples_leaf=19,
                                                    min_samples_split=15,
                                                    n_jobs=1, random_state=42),
                     step=0.9588310222438)),
                ('featureunion-1',
                 FeatureUnion(transformer_list=[('skiptransformer',
                                                 SkipTransformer()),
                                                ('passthrough',
                                                 Passthrough())])),
                ('featureunion-2',
                 FeatureUnion(transformer_lis

,precision,recall,f1-score,support
nothing,0.588,0.455,0.513,22.000
personality_cult,0.375,0.429,0.400,7.000
vilifying_eu,0.000,0.000,0.000,12.000
vilifying_neighbours,0.457,0.800,0.582,20.000
vilifying_opponents,0.806,0.817,0.811,71.000
accuracy,0.659,0.659,0.659,0.659
macro avg,0.445,0.500,0.461,132.000
weighted avg,0.620,0.659,0.631,132.000
